In [1]:
from typing import List
import pickle
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from cv2.datasets import none
from narwhals import Boolean
from skimage.measure import fit
import time

In [2]:

def open_ghost_data(file_name: str):
	inputs = []
	expected_outputs = []
	with open(file_name, "r", encoding="utf-8") as file:
		for line in file:
			parsed_line = line.split()

			if len(parsed_line) >= 4:
				inputs.append([float(x) for x in parsed_line[0:3]])
				temp_output = [0, 0, 0, 0]
				temp_output[int(parsed_line[3]) - 1] = 1
				expected_outputs.append(temp_output)
	return np.array(inputs), np.array(expected_outputs)


def transform_mnist_data(y):
	output = []
	for data in y:
		temp = [0 for _ in range(10)]
		temp[int(data) - 1] = 1
		output.append(temp)
	return np.array(output)




In [3]:

train_inputs, train_outputs = open_ghost_data("../data/colors_train.txt")
test_inputs, test_outputs = open_ghost_data("../data/colors_test.txt")


In [4]:
import pandas
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
x_data, y_data = mnist.data, transform_mnist_data(mnist.target)
x_data = x_data / 255

In [5]:


x_train, x_test = x_data[:60000], x_data[60000:]
y_train, y_test = y_data[:60000], y_data[60000:]
print(f"x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"x_test: {x_test.shape}, y_test: {y_test.shape}")

x_train: (60000, 784), y_train: (60000, 10)
x_test: (10000, 784), y_test: (10000, 10)


In [6]:

class Module:
	def forward(self, x):
		raise NotImplementedError

	def backward(self, grad):
		raise NotImplementedError

	def adjust(self, eta):
		raise NotImplementedError


class PoolLayer(Module):
	def __init__(self, mask_size, step, flatten: bool = False):
		self.mask_size = mask_size
		self.step = step
		self.cache_shape = None
		self.cache_argmax = None
		self.is_1d = False
		self.flatten = flatten

	def forward(self, x):
		num_windows, kernels = x.shape
		side = int(np.sqrt(num_windows))
		x_3d = x.T.reshape(kernels, side, side)

		windows = np.lib.stride_tricks.sliding_window_view(x_3d, window_shape=(self.mask_size, self.mask_size),
		                                                   axis=(1, 2))

		windows_strided = windows[:, ::self.step, ::self.step]
		out_h, out_w = windows_strided.shape[1:3]
		flat_windows = windows_strided.reshape(kernels, out_h, out_w, -1)
		self.cache_argmax = np.argmax(flat_windows, axis=-1)
		output = np.max(flat_windows, axis=-1)

		if self.flatten:
			return output.flatten()

		return output.reshape(kernels, -1).T

	def backward(self, grad: np.ndarray):

		# 1. Przywracamy gradient do kształtu (kernels, out_h, out_w)
		if self.flatten:
			# Jeśli w forward był flatten(), grad przyszedł jako 1D wektor
			kernels = self.cache_argmax.shape[0]
			out_h, out_w = self.cache_argmax.shape[1:3]
			grad = grad.reshape(kernels, out_h, out_w)
		else:
			# Jeśli w forward poszedł (num_windows, kernels).T
			# grad przychodzi jako (out_h * out_w, kernels) -> obracamy na (kernels, out_h, out_w)
			grad = grad.T.reshape(self.cache_argmax.shape[:3])

		kernels, out_h, out_w = grad.shape
		mask_sq = self.mask_size * self.mask_size

		# 2. Tworzymy pustą macierz błędów 1D dla każdego okienka poolingowego
		# Kształt: (kernels, out_h, out_w, mask_size * mask_size)
		grad_flat = np.zeros((kernels, out_h, out_w, mask_sq), dtype=grad.dtype)

		# 3. Rozstawiamy indeksy dla np.put_along_axis
		# Tworzymy siatkę indeksów 4D, aby wskazać dokładnie wygrane pozycje z argmax
		k_idx, i_idx, j_idx = np.ogrid[:kernels, :out_h, :out_w]

		# Wstawiamy gradienty dokladnie w miejsca max wartości (zapisane w cache_argmax)
		np.put_along_axis(grad_flat, self.cache_argmax[..., None], grad[..., None], axis=-1)

		# 4. Przywracamy kształt okienek (kernels, out_h, out_w, mask_size, mask_size)
		grad_windows = grad_flat.reshape(kernels, out_h, out_w, self.mask_size, self.mask_size)

		# 5. Odtwarzamy pełne płótno dla wejścia z Conv (kernels, side, side)
		# Odtwarzamy wymiar 'side' na podstawie cache_argmax i stride
		side = (out_h - 1) * self.step + self.mask_size
		grad_x_3d = np.zeros((kernels, side, side), dtype=grad.dtype)

		# Genialna wektoryzacja rozrzucania gradientu (col2im)
		# Generujemy siatkę nakładania się masek poolingowych
		i_starts = np.arange(out_h) * self.step
		j_starts = np.arange(out_w) * self.step

		i_idx_full = i_starts.reshape(1, -1, 1, 1, 1) + np.arange(self.mask_size).reshape(1, 1, 1, -1, 1)
		j_idx_full = j_starts.reshape(1, 1, -1, 1, 1) + np.arange(self.mask_size).reshape(1, 1, 1, 1, -1)
		k_idx_full = np.arange(kernels).reshape(-1, 1, 1, 1, 1)

		flat_indices = (k_idx_full * side * side + i_idx_full * side + j_idx_full).ravel()

		# Sumujemy błędy bez pętli za pomocą np.add.at
		np.add.at(grad_x_3d.ravel(), flat_indices, grad_windows.ravel())

		# 6. Zwracamy gradient z powrotem w formacie (num_windows, kernels)
		# Idealnie dopasowanym do backward w ConvLayer!
		return grad_x_3d.reshape(kernels, -1).T

	def adjust(self, eta):
		pass


class LinearLayer(Module):
	def __init__(self, input_size: int, output_size: int, weights: None | np.ndarray = None,
				 biases: None | np.ndarray = None, min_weight: None | float = None, max_weight: None | float = None,
				 dropout_rate: float = 0.0):
		if weights is not None:
			if weights.shape != (output_size, input_size):
				raise ValueError("niezgodne wymiary macierzy wag")
			self.W = np.copy(weights)
		else:
			self.randomize_weights(min_weight, max_weight, output_size, input_size)

		if biases is not None:
			biases = np.atleast_2d(biases)
			if biases.shape != (1, output_size):
				raise ValueError("niezgodne wymiary macierzy biasow")
			self.b = np.copy(biases)
		else:
			self.b = None
		self.cache_X = None
		self.cache_grad = None
		self.is_training = False
		self.dropout_rate = dropout_rate
		self.dropout_mask = None

	def randomize_weights(self, min, max, output_size, input_size):
		self.W = np.random.uniform(min, max, (output_size, input_size))

	def forward(self, x):
		self.cache_X = np.atleast_2d(x)

		output = x @ self.W.T.copy()

		if self.b is not None:
			return output + self.b

		if self.dropout_rate > 0 and self.is_training:
			keep_rate = 1 - self.dropout_rate
			self.dropout_mask = np.random.binomial(1, keep_rate, output.shape) * (1 / keep_rate)
			output = output * self.dropout_mask
		else:
			self.dropout_mask = None
		return output

	def backward(self, grad):
		if self.dropout_mask is not None:
			grad = grad * self.dropout_mask
		self.cache_grad = np.atleast_2d(grad)
		return grad @ self.W

	def adjust(self, eta):
		grad_W = self.cache_grad.T @ self.cache_X
		self.W -= eta * grad_W
		if self.b is not None:
			grad_b = np.sum(self.cache_grad, axis=0, keepdims=True)
			self.b -= eta * grad_b


class ActivationLayer(Module):

	@staticmethod
	def relu(x: np.ndarray) -> np.ndarray:
		return np.maximum(x, 0)

	@staticmethod
	def relu_derivative(x: np.ndarray) -> np.ndarray:
		return np.where(x > 0, 1.0, 0)

	@staticmethod
	def sigmoid(x: np.ndarray) -> np.ndarray:
		return 1 / (1 + np.exp(-x))

	@staticmethod
	def sigmoid_derivative(x: np.ndarray) -> np.ndarray:
		return x * (1 - x)

	@staticmethod
	def tanh(x: np.ndarray) -> np.ndarray:
		return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

	@staticmethod
	def tanh_derivative(x: np.ndarray) -> np.ndarray:
		return 1 - x ** 2

	@staticmethod
	def softmax(x: np.ndarray) -> np.ndarray:
		exps = np.exp(x - np.max(x, axis=1, keepdims=True))
		return exps / np.sum(exps, axis=1, keepdims=True)

	ACTIVATION_FUNCTIONS = {
		'ReLU': (relu, relu_derivative),
		'Sigmoid': (sigmoid, sigmoid_derivative),
		'Tanh': (tanh, tanh_derivative),
		'Softmax': (softmax, None)
	}

	def __init__(self, activation_function):
		self.activation_name = activation_function
		self.activation_function = self.ACTIVATION_FUNCTIONS[activation_function][0]
		self.derivative_function = self.ACTIVATION_FUNCTIONS[activation_function][1]
		self.cache_output = None

	def forward(self, S):
		output = self.activation_function(S)
		self.cache_output = output
		return output

	def backward(self, grad):
		if self.activation_name == 'Softmax':
			return self.cache_output * (grad - np.sum(grad * self.cache_output, axis=1, keepdims=True))

		return grad * self.derivative_function(self.cache_output)

	def adjust(self, eta):
		pass


class ConvolutionalLayer(Module):
	def __init__(self, filter_list: list[np.ndarray], conv_step: int, padding: int, flatten: bool = False):
		weights = []
		for filter in filter_list:
			weights.append(filter.reshape((np.pow(filter.shape[0], 2))))
		weights = np.array(weights)

		self.conv_step = conv_step
		self.padding = padding
		self.kernel_count = weights.shape[0]
		self.matrix_size = int(np.sqrt(weights.shape[1]))
		self.W = weights
		self.cache_X = None
		self.cache_grad = None
		self.original_shape = None
		self.padded_shape = None
		self.flattening = flatten
		self.cache_out_shape = None

	def forward(self, x: np.ndarray):
		if x.ndim == 1 or (x.ndim == 2 and x.shape[0] == 1):
			side = int(np.sqrt(x.size))
			x = x.reshape((side, side))

		self.original_shape = x.shape
		padded_image = np.pad(x, self.padding, mode='constant', constant_values=0)
		self.padded_shape = padded_image.shape

		windows = np.lib.stride_tricks.sliding_window_view(padded_image,
														   window_shape=(self.matrix_size, self.matrix_size))

		windows_strided = windows[::self.conv_step, ::self.conv_step, :, :]

		out_h, out_w = windows_strided.shape[:2]
		self.cache_out_shape = (out_h, out_w)
		padded_w = self.padded_shape[1]
		i_starts = np.arange(out_h) * self.conv_step
		j_starts = np.arange(out_w) * self.conv_step

		i_idx = i_starts.reshape(-1, 1, 1, 1) + np.arange(self.matrix_size).reshape(1, 1, -1, 1)
		j_idx = j_starts.reshape(1, -1, 1, 1) + np.arange(self.matrix_size).reshape(1, 1, 1, -1)

		self.cache_flat_indices = i_idx * padded_w + j_idx

		self.cache_X = windows_strided.transpose((0, 1, 3, 2)).reshape(-1, self.matrix_size ** 2)

		output = self.cache_X @ self.W.T
		if self.flattening:
			output = output.flatten()

		return output

	def backward(self, grad: np.ndarray):
		num_fragments = self.cache_X.shape[0]
		grad_reshaped = np.reshape(grad, (num_fragments, self.kernel_count))
		self.cache_grad = grad_reshaped
		grad_X_frag = self.cache_grad @ self.W

		padded_h, padded_w = self.padded_shape
		out_h, out_w = self.cache_out_shape

		grad_X_windows = grad_X_frag.reshape(out_h, out_w, self.matrix_size, self.matrix_size)
		grad_X_windows = grad_X_windows.transpose(0, 1, 3, 2)

		grad_padded_flat = np.zeros(padded_h * padded_w)
		np.add.at(grad_padded_flat, self.cache_flat_indices.ravel(), grad_X_windows.ravel())

		grad_padded = grad_padded_flat.reshape(padded_h, padded_w)

		grad_out = grad_padded[
			self.padding: self.padding + self.original_shape[0],
			self.padding: self.padding + self.original_shape[1]
		]

		if hasattr(self, 'input_was_flat') and self.input_was_flat:
			return grad_out.flatten()

		return grad_out

	def adjust(self, eta):
		grad_W = self.cache_grad.T @ self.cache_X
		self.W -= eta * grad_W


class NeuralNetworkException(Exception):
	def __init__(self, msg):
		self.messege = msg

	def get_message(self):
		return self.messege


class NeuralNetwork:

	def __init__(self):
		self.layers: list[Module] = list()

	def add_layer(self, layer: Module):
		self.layers.append(layer)

	def save_model(self, filename: str):
		params = {}
		for i, layer in enumerate(self.layers):
			if isinstance(layer, LinearLayer):
				params[f'W_{i}'] = layer.W
				if layer.b is not None:
					params[f'b_{i}'] = layer.b
		np.savez(filename, **params)

	def load_model(self, filename: str):
		data = np.load(filename if filename.endswith('.npz') else filename + '.npz')
		for i, layer in enumerate(self.layers):
			if isinstance(layer, LinearLayer):
				layer.W = data[f'W_{i}']
				if f'b_{i}' in data:
					layer.b = data[f'b_{i}']

	def predict(self, x: np.ndarray | None):
		output = x
		for layer in self.layers:
			output = layer.forward(output)

		return output

	def train_step(self, x, y, learning_rate) -> float:
		prediction = self.predict(x)
		error = prediction - y
		loss = float(np.mean(np.square(error)))
		grad = (2.0 / error.size) * error
		for layer in reversed(self.layers):
			grad = layer.backward(grad)

		for layer in self.layers:
			layer.adjust(learning_rate)

		return loss

	def fit(self, x: np.ndarray, y, epochs, batch_size, learning_rate):
		self.switch_training(True)
		for epoch in range(epochs):
			print(epoch)
			indices = np.arange(x.shape[0])
			np.random.shuffle(indices)
			x_shuffled = x[indices]
			y_shuffled = y[indices]
			epoch_loss = 0.0
			batches_count = 0
			for batch in range(0, x.shape[0], batch_size):
				loss = self.train_step(x_shuffled[batch:batch + batch_size], y_shuffled[batch:batch + batch_size],
									   learning_rate)
				epoch_loss += loss
				batches_count += 1

		self.switch_training(False)

	pass

	def switch_training(self, state: bool):
		for layer in self.layers:
			if hasattr(layer, 'is_training'):
				layer.is_training = state

	def evaluate(self, x, y):
		data_size = y.shape[0]
		correct_predictions_count = 0
		false = []
		for i in range(data_size):
			predicted_num = np.argmax(self.predict(x[i:i + 1]))
			expected_num = np.argmax(y[i])
			if int(predicted_num) == int(expected_num):
				correct_predictions_count += 1
			else:
				false.append(i)
		return ((correct_predictions_count / data_size), false)

In [7]:
import time
import json
import numpy as np


class ExperimentRunner:
	def __init__(self, x_data, y_data, x_test, y_test, save_dir="../saves/"):
		self.x_data = x_data
		self.y_data = y_data
		self.x_test = x_test
		self.y_test = y_test
		self.save_dir = save_dir
		self.results = []

	def run_experiment(self, test_id: int, model: NeuralNetwork, test_params: dict):
		print(f"\n{'=' * 40}")
		print(f"TEST {test_id}")
		print(f"Data: {test_params['training_data_amount']}, Epochs: {test_params['iterations']}, "
			  f"Batch: {test_params['batch_size']}, LR: {test_params['alpha']}")
		print(f"{'=' * 40}")

		current_x_train = self.x_data[:test_params['training_data_amount']]
		current_y_train = self.y_data[:test_params['training_data_amount']]

		current_x_test = self.x_test[:test_params['test_data_amount']]
		current_y_test = self.y_test[:test_params['test_data_amount']]

		start_time = time.perf_counter()
		model.fit(current_x_train, current_y_train,
				  epochs=test_params['iterations'],
				  learning_rate=test_params['alpha'],
				  batch_size=test_params['batch_size'])
		end_time = time.perf_counter()
		training_time = end_time - start_time

		accuracy, _ = model.evaluate(current_x_test, current_y_test)

		print(f"Training Time: {training_time:.2f}s")
		print(f"Test Accuracy: {accuracy * 100:.2f}%")

		filename = f"{self.save_dir}model_test_{test_id}_data_{test_params['training_data_amount']}.npz"
		model.save_model(filename)
		print(f"Weights saved: {filename}")

		self._record_result(test_id, test_params, training_time, accuracy)

	def _record_result(self, test_id, test_params, time_taken, accuracy):
		result_entry = {
			"test_id": test_id,
			"accuracy": round(accuracy * 100, 2),
			"time_seconds": round(time_taken, 2),
			**test_params
		}
		self.results.append(result_entry)

	def save_summary_to_json(self, filename="experiment_results.json"):
		with open(filename, 'w') as f:
			json.dump(self.results, f, indent=4)
		print(f"\nZ: {filename}")

In [9]:
#zad 1 i 2

Wi_size = x_data.shape[1]
tests_parameters = [
	{
		'Wh_neurons': 40,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.005,
		'min_weight': -0.1,
		'max_weight': 0.1,
		'iterations': 100,
		'batch_size': 1
	}
	,
	{
		'Wh_neurons': 100,
		'training_data_amount': 5000,
		'test_data_amount': 10000,
		'alpha': 0.005,
		'min_weight': -0.1,
		'max_weight': 0.1,
		'iterations': 100,
		'batch_size': 1
	}
	,
	{
		'Wh_neurons': 100,
		'training_data_amount': 15000,
		'test_data_amount': 10000,
		'alpha': 0.005,
		'min_weight': -0.1,
		'max_weight': 0.1,
		'iterations': 100,
		'batch_size': 1
	}
	, #zadanie 2
	{
		'Wh_neurons': 40,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.1,
		'min_weight': -0.1,
		'max_weight': 0.1,
		'iterations': 100,
		'batch_size': 100
	}
	,
	{
		'Wh_neurons': 100,
		'training_data_amount': 5000,
		'test_data_amount': 10000,
		'alpha': 0.1,
		'min_weight': -0.1,
		'max_weight': 0.1,
		'iterations': 100,
		'batch_size': 100
	}
	,
	{
		'Wh_neurons': 100,
		'training_data_amount': 15000,
		'test_data_amount': 10000,
		'alpha': 0.1,
		'min_weight': -0.1,
		'max_weight': 0.1,
		'iterations': 100,
		'batch_size': 100
	}
]

runner = ExperimentRunner(x_data, y_data, x_test, y_test, save_dir="../saves/")

for i, test in enumerate(tests_parameters):

	test_network = NeuralNetwork()

	test_network.add_layer(LinearLayer(Wi_size, test['Wh_neurons'],
									   min_weight=test['min_weight'],
									   max_weight=test['max_weight'],
									   dropout_rate=0.5))
	test_network.add_layer(ActivationLayer("ReLU"))

	test_network.add_layer(LinearLayer(test['Wh_neurons'], y_test.shape[1],
									   min_weight=test['min_weight'],
									   max_weight=test['max_weight'],
									   dropout_rate=0.0))

	runner.run_experiment(test_id=i + 1, model=test_network, test_params=test)

runner.save_summary_to_json("wyniki_sieci_mnist_Z4_1-2.json")


TEST 1
Data: 1000, Epochs: 100, Batch: 1, LR: 0.005
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
Training Time: 38.19s
Test Accuracy: 85.33%
Weights saved: ../saves/model_test_1_data_1000.npz

TEST 2
Data: 5000, Epochs: 100, Batch: 1, LR: 0.005
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
Training Time: 310.43s
Test Accuracy: 92.48%
Weights saved: ../saves/model_test_2_data_5000.npz

TEST 3
Data: 15000, Epochs: 100, Batch: 1, LR: 0.005
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
2

In [11]:

Wi_size = x_data.shape[1]
tests_parameters = [
	{
		'Wh_neurons': 100,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.01,
		'min_weight': -0.01,
		'max_weight': 0.01,
		'iterations': 100,
		'batch_size': 100
	},

		{
		'Wh_neurons': 100,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.005,
		'min_weight': -0.01,
		'max_weight': 0.01,
		'iterations': 100,
		'batch_size': 100
	},
		{
		'Wh_neurons': 100,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.02,
		'min_weight': -0.01,
		'max_weight': 0.01,
		'iterations': 100,
		'batch_size': 100
	},
		{
		'Wh_neurons': 100,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.04,
		'min_weight': -0.01,
		'max_weight': 0.01,
		'iterations': 100,
		'batch_size': 100
	}
,
		{
		'Wh_neurons': 100,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.08,
		'min_weight': -0.01,
		'max_weight': 0.01,
		'iterations': 100,
		'batch_size': 100
	}
	,
		{
		'Wh_neurons': 100,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.12,
		'min_weight': -0.01,
		'max_weight': 0.01,
		'iterations': 100,
		'batch_size': 100
	}
	,
		{
		'Wh_neurons': 100,
		'training_data_amount': 60000,
		'test_data_amount': 10000,
		'alpha': 0.08,
		'min_weight': -0.01,
		'max_weight': 0.01,
		'iterations': 100,
		'batch_size': 100
	}

]

runner = ExperimentRunner(x_data, y_data, x_test, y_test, save_dir="../saves/")

for i, test in enumerate(tests_parameters):



	test_network = NeuralNetwork()

	test_network.add_layer(LinearLayer(Wi_size, test['Wh_neurons'],
									   min_weight=test['min_weight'],
									   max_weight=test['max_weight'],
									   dropout_rate=0.5))
	test_network.add_layer(ActivationLayer("Tanh"))

	test_network.add_layer(LinearLayer(test['Wh_neurons'], y_test.shape[1],
									   min_weight=test['min_weight'],
									   max_weight=test['max_weight'],
									   dropout_rate=0.0))
	test_network.add_layer(ActivationLayer("Softmax"))

	runner.run_experiment(test_id=i + 1, model=test_network, test_params=test)

runner.save_summary_to_json("wyniki_sieci_mnist_Z4_3.json")




TEST 1
Data: 1000, Epochs: 100, Batch: 100, LR: 0.01
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
Training Time: 15.67s
Test Accuracy: 22.41%
Weights saved: ../saves/model_test_1_data_1000.npz

TEST 2
Data: 1000, Epochs: 100, Batch: 100, LR: 0.005
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
Training Time: 6.72s
Test Accuracy: 17.99%
Weights saved: ../saves/model_test_2_data_1000.npz

TEST 3
Data: 1000, Epochs: 100, Batch: 100, LR: 0.02
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
